# Topic 3A: Linear Regression for Loss Given Default (LGD)
**Module 1 - Introduction to Machine Learning in Python**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from scipy.special import expit  # Sigmoid function


## 1. Create Synthetic LGD Dataset


In [ ]:
np.random.seed(42)
n = 10000
df = pd.DataFrame({
    'loan_to_value': np.random.uniform(0.4, 1.3, n),
    'collateral_ratio': np.random.uniform(0.2, 1.5, n),
    'borrower_income': np.random.lognormal(10.5, 0.8, n),
    'loan_age_months': np.random.randint(6, 120, n),
    'secured': np.random.binomial(1, 0.6, n),
    'gdp_growth': np.random.normal(0.02, 0.015, n),
})

# Generate LGD target (bounded 0-1)
logit = (-2 + 1.5*df['loan_to_value'] - 0.8*df['collateral_ratio']
         - 0.5*df['secured'] - 5*df['gdp_growth']
         + np.random.normal(0, 0.5, n))
df['lgd'] = expit(logit)

print(f'LGD statistics:')
print(df['lgd'].describe().round(4))


## 2. Explore the Target Variable


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['lgd'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('LGD Distribution (Raw)')
axes[0].set_xlabel('Loss Given Default')

# Logit transform for modeling
df['lgd_logit'] = np.log(df['lgd'].clip(0.001, 0.999) / (1 - df['lgd'].clip(0.001, 0.999)))
axes[1].hist(df['lgd_logit'], bins=50, color='indianred', edgecolor='white')
axes[1].set_title('LGD Distribution (Logit-Transformed)')
axes[1].set_xlabel('Logit(LGD)')
plt.tight_layout()
plt.show()


## 3. Train and Compare Regression Models


In [ ]:
features = ['loan_to_value', 'collateral_ratio', 'borrower_income', 'loan_age_months', 'secured', 'gdp_growth']
X = df[features]
y = df['lgd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'OLS': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Ridge (alpha=10)': Ridge(alpha=10.0),
    'Lasso (alpha=0.01)': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
}

results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s).clip(0, 1)  # Clip to valid range
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    n_nonzero = np.sum(model.coef_ != 0) if hasattr(model, 'coef_') else 'N/A'
    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Non-zero coefs': n_nonzero})

print(pd.DataFrame(results).round(4).to_string(index=False))


## 4. Coefficient Interpretation


In [ ]:
best_model = Ridge(alpha=1.0).fit(X_train_s, y_train)
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': best_model.coef_,
}).sort_values('Coefficient', key=abs, ascending=False)

print('Ridge Regression Coefficients:')
print(coef_df.round(4).to_string(index=False))
print(f'\nIntercept: {best_model.intercept_:.4f}')
print('\nInterpretation: A positive coefficient means higher values of that feature')
print('are associated with higher LGD (more loss).')


## 5. Residual Diagnostics


In [ ]:
y_pred = best_model.predict(X_test_s).clip(0, 1)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residuals vs Fitted
axes[0].scatter(y_pred, residuals, alpha=0.1, s=5)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Histogram of residuals
axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Distribution of Residuals')

# Q-Q plot (approximate)
from scipy import stats
stats.probplot(residuals, plot=axes[2])
axes[2].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()
